# LLM-project pipeline

Three-step flow: 1. RAG test → 2. Train and save → 3. Inference.
Run cells in order, or run only the step you need. Config is loaded from `qwen_math_flow/hyperparameters.py`; you can override variables in the next cell.

In [2]:
import json
from pathlib import Path

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from qwen_math_flow.download_model import download_qwen_25_07b
from qwen_math_flow.external_calculator import SafeEvalCalculatorClient, StubCalculatorClient
from qwen_math_flow.load_dataset import format_gsm8k_as_chat, load_and_tokenize_math
from qwen_math_flow.lora_finetune import create_lora_model, run_finetune
from qwen_math_flow.rag_calculator import RAGCalculatorLayer

## 1. RAG test (no model)

In [2]:
from qwen_math_flow.hyperparameters import USE_SAFE_EVAL_RAG_TEST

client = SafeEvalCalculatorClient() if USE_SAFE_EVAL_RAG_TEST else StubCalculatorClient()
rag = RAGCalculatorLayer(client)
tests = [
    "Result: [CALC: 2+3*4]",
    "First compute [CALC: 10/2], then add 5.",
]
print("RAG test (augment only, no model):")
for s in tests:
    out, pairs = rag.augment(s, inject_into_context=True)
    print(f"  in:  {s}")
    print(f"  out: {out}")
    print(f"  calc: {pairs}")
print("RAG test done.\n")

RAG test (augment only, no model):
  in:  Result: [CALC: 2+3*4]
  out: Result: 14
  calc: [('2+3*4', '14')]
  in:  First compute [CALC: 10/2], then add 5.
  out: First compute 5.0, then add 5.
  calc: [('10/2', '5.0')]
RAG test done.



## 2. Train and save

In [3]:
from qwen_math_flow.hyperparameters import (
    ADAPTER_DIR,
    DATASET_NAME,
    DATASET_SPLIT,
    GRADIENT_ACCUMULATION_STEPS,
    LEARNING_RATE,
    LOAD_IN_4BIT,
    LORA_ALPHA,
    LORA_R,
    MAX_LENGTH,
    MAX_TRAIN_SAMPLES,
    MODEL_CACHE_DIR,
    NUM_EPOCHS,
    PER_DEVICE_TRAIN_BATCH_SIZE,
)

model, tokenizer = download_qwen_25_07b(
    cache_dir=MODEL_CACHE_DIR,
    load_in_4bit=LOAD_IN_4BIT,
    device_map="auto" if LOAD_IN_4BIT else None,
)
tokenized_train = load_and_tokenize_math(
    tokenizer,
    name=DATASET_NAME,
    split=DATASET_SPLIT,
    max_samples=MAX_TRAIN_SAMPLES,
    max_length=MAX_LENGTH,
    message_formatter=format_gsm8k_as_chat,
)
cap_msg = "full train split" if MAX_TRAIN_SAMPLES is None else f"up to {MAX_TRAIN_SAMPLES} samples"
print(f"GSM8K training: {len(tokenized_train)} samples ({cap_msg}), {NUM_EPOCHS} epoch(s).")
peft_model = create_lora_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    use_4bit_or_8bit=LOAD_IN_4BIT,
)
run_finetune(
    peft_model,
    tokenizer,
    tokenized_train,
    output_dir=ADAPTER_DIR,
    num_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
)
print(f"Train done. Adapter + tokenizer saved to: {ADAPTER_DIR}\n")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Tokenizing (num_proc=1):   0%|          | 0/100 [00:00<?, ? examples/s]

GSM8K training: 100 samples (up to 100 samples), 2 epoch(s).


Step,Training Loss
10,1.442348


Train done. Adapter + tokenizer saved to: output/lora_math



## 3. Inference

In [3]:
import random
from datasets import load_dataset

from qwen_math_flow.hyperparameters import (
    ADAPTER_DIR,
    INFERENCE_NUM_QUESTIONS,
    INFERENCE_QUESTION_SPLIT,
    INFERENCE_RANDOM_SEED,
    LOAD_IN_4BIT_INFERENCE,
    MAX_NEW_TOKENS,
)

adapter_path = Path(ADAPTER_DIR)
with open(adapter_path / "adapter_config.json") as f:
    base_model_id = json.load(f).get("base_model_name_or_path", "Qwen/Qwen2.5-0.5B-Instruct")

tokenizer = AutoTokenizer.from_pretrained(str(adapter_path), trust_remote_code=True)
base_kwargs = dict(device_map="auto", trust_remote_code=True)
if LOAD_IN_4BIT_INFERENCE:
    base_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
    )
else:
    base_kwargs["torch_dtype"] = "auto"

base_model = AutoModelForCausalLM.from_pretrained(base_model_id, **base_kwargs)
model = PeftModel.from_pretrained(base_model, str(adapter_path))
model.eval()

rag = RAGCalculatorLayer(SafeEvalCalculatorClient())

raw = load_dataset("openai/gsm8k", "main", split=INFERENCE_QUESTION_SPLIT)
rng = random.Random(INFERENCE_RANDOM_SEED)
n = min(INFERENCE_NUM_QUESTIONS, len(raw))
indices = rng.sample(range(len(raw)), n)

for i, idx in enumerate(indices, start=1):
    question = raw[idx]["question"]
    user_content = f"Solve the following math problem step by step.\n\n{question}"
    messages = [{"role": "user", "content": user_content}]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    answer = rag.generate_with_rag(model, tokenizer, prompt, max_new_tokens=MAX_NEW_TOKENS)
    print(f"--- [{i}/{n}] (GSM8K {INFERENCE_QUESTION_SPLIT} index {idx}) ---")
    print(f"Question: {question}\n")
    print(f"Answer: {answer}\n")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


--- [1/5] (GSM8K test index 1309) ---
Question: The girls are trying to raise money for a carnival. Kim raises $320 more than Alexandra, who raises $430, and Maryam raises $400 more than Sarah, who raises $300. How much money, in dollars, did they all raise in total?

Answer: system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Solve the following math problem step by step.

The girls are trying to raise money for a carnival. Kim raises $320 more than Alexandra, who raises $430, and Maryam raises $400 more than Sarah, who raises $300. How much money, in dollars, did they all raise in total?
assistant
To determine the total amount of money raised by the girls, we need to calculate the individual amounts raised by each girl and then sum them up.

1. **Kim's Amount:**
   - Kim raises $320 more than Alexandra.
   - Alexandra raises $430.
   - Therefore, Kim raises:
     \[
     430 + 320 = 750 \text{ dollars}
     \]

2. **Alexandra's Amount:**
   - Alexandra ra